# U03-A05 — Oficina — Primeira base processável

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/04_oficina_base_processavel.ipynb)

Este notebook é um roteiro de projeto. Execute transformações em uma cópia
própria ou notebook técnico; registre aqui decisões, evidências e resultados.
Não altere os arquivos brutos.

![Dados brutos, código, dados derivados e documentação formam um pacote submetido a testes e parecer de qualidade, com retorno rastreável para correções.](imagens/04_pacote_processavel.svg)

A figura funciona também como checklist: um CSV isolado não demonstra como a
base foi produzida, testada ou delimitada. A entrega precisa combinar os
componentes e manter vínculos entre eles.

A oficina começa pelo inventário porque nenhuma transformação pode ser
avaliada sem conhecer pergunta, unidade, entradas, versões e política de
preservação. Essa descrição estabelece o escopo de todas as etapas seguintes.

## 1. Inventário e estrutura

**Pergunta e unidade de análise:** no recorte de 10.454 livros do Goodreads,
livros de autoria registrada como feminina recebem número de avaliações e de
resenhas semelhante ao de livros de autoria registrada como masculina, e a
diferença muda conforme o gênero literário? Unidade: o livro.

**Arquivos, formatos, versões e proveniência:** `livros_good_reads.csv` (CSV,
UTF-8 com BOM, SHA-256 `40effdd6…b9cfe64`, recebido em 15/10/2026);
`proveniencia_goodreads.json` (origem declarada e lacunas);
`dicionario_dados.csv` (15 campos documentados).

**Estrutura de pastas e política de imutabilidade:**
`projeto_goodreads/dados/brutos/` (somente leitura, nunca editado);
`dados/intermediarios/livros_normalizado.csv` (originais e colunas
normalizadas lado a lado); `dados/derivados/` (tabelas para análise e log). Os
arquivos intermediários e derivados são sempre reconstruídos pelo código e
nunca editados à mão.

**Ambiente e dependências:** Python 3 com pandas 2.x e numpy (versões gravadas
em `log_execucao.json`); `gender-guesser` 0.4.0 para a auditoria por nome (se
ausente, o código usa a auditoria gravada); bibliotecas do `requirements.txt`
da disciplina.

Com as entradas e o ambiente identificados, podemos registrar como cada fonte
será convertida em uma representação legível. As escolhas de importação e
extração precisam produzir saídas compatíveis com o inventário declarado.

## 2. Importação e extração

| Fonte | Leitor/parâmetros | Estrutura esperada | Teste | Saída |
|---|---|---|---|---|
| `livros_good_reads.csv` | `pd.read_csv(sep=",", encoding="utf-8-sig")` | 10.454 × 15, colunas na ordem documentada | hash, colunas, unicidade de `id_livro`, ausência de valores vazios | DataFrame bruto em memória |
| `proveniencia_goodreads.json` | `json.loads` | objeto com chaves de origem e lacunas | conferência do hash | metadados do pacote |
| `dicionario_dados.csv` | `pd.read_csv` | 15 linhas, uma por campo | campos documentados = campos observados | documentação |

**PDFs com texto, PDFs de imagem e TXT:** não há no projeto. Títulos e nomes
chegam como texto digital no CSV.

**Plano de OCR, amostra de controle e métrica de erro:** não se aplica, porque
não há imagem. O equivalente no projeto é a **amostra de controle do rótulo de
gênero**: os 12 autores mais avaliados de cada sentido de divergência entre
rótulo e nome, conferidos com conhecimento público, e a taxa de divergência por
categoria de prenome como métrica de erro do rótulo.

A importação produz objetos técnicos, mas ainda não garante uma organização
coerente para análise. O modelo tabular define o que cada linha representa e
quais transformações serão aplicadas sem apagar os valores recebidos.

## 3. Modelo tabular e transformação

**Tabelas e unidade de cada linha:** `livros` (um livro), `autores` (um
autor), `livro_generos` (uma relação livro–gênero literário),
`auditoria_genero_nome` (um autor) e `base_analitica` (um livro com atributos
do autor).

**Decisão largo/longo:** os gêneros literários chegam em formato largo
(`genero_1`, `genero_2`). Mantenho `genero_1` na base analítica, porque cada
livro precisa entrar uma única vez nas comparações principais, e crio
`livro_generos` em formato longo para análises que considerem os dois rótulos.

| Campo | Original preservado | Regra | Justificativa | Teste | Perda possível |
|---|---|---|---|---|---|
| `titulo_chave` | `titulo` | remove série entre parênteses, acentos, caixa e pontuação; mantém letras Unicode | detectar duplicatas | comparação com regra ASCII | títulos distintos com mesma chave |
| `pais_padronizado` | `pais_original` | tabela de equivalências; `regra_pais` | agrupar por país | todos os valores recebem regra | nações do Reino Unido agregadas |
| `periodo_publicacao` | `ano_original` | faixas: antes de 1900, 1900–1989, 1990–2009, 2010–2019 | comparar épocas | contagem por faixa | detalhes dentro de cada faixa |
| `taxa_resenhas` | `n_resenhas`, `n_avaliacoes` | razão | engajamento relativo | valores entre 0 e 1 | tamanho absoluto |
| `genero_E2_sem_duvidosos` | `genero_autor_original` | ausente se rótulo “masculino” e prenome feminino | evitar grupo contaminado | contagem de duvidosos | 1.274 livros fora da análise principal |

As regras do modelo tabular tornam valores comparáveis e auditáveis. Essa
comparabilidade permite diagnosticar ausências e semelhanças, mas não autoriza
preencher lacunas ou eliminar registros automaticamente.

## 4. Ausências e duplicatas

**Representações de ausência e razões:** o arquivo bruto não tem células
vazias. Ausências criadas pela padronização ficam como valor ausente em
`pais_padronizado`, com a razão em `regra_pais` (ambíguo, URL, residência em
vez de nascimento etc.). Na especificação E2 de gênero, o valor ausente
significa “rótulo duvidoso”, não “gênero desconhecido”.

**Chave de duplicata exata:** `id_livro` (nenhuma repetição).

**Critérios de possível duplicata e revisão humana:** mesma `titulo_chave` e
mesmo `id_autor`: nenhum caso. Mesma `titulo_chave` com autores diferentes: 18
livros, revisados e mantidos como obras distintas (inclui uma adaptação).

**Decisão de manter, relacionar, fundir ou excluir:** manter todos os livros;
não fundir nenhum registro; excluir apenas em análises de sensibilidade
documentadas.

Depois de representar ausências e revisar possíveis duplicatas, as unidades
estão mais bem definidas para integração. Agora é necessário declarar como as
tabelas se relacionam e testar se a junção preserva o número e a identidade dos casos.

## 5. Junções e integração

| Tabelas | Chave | Cardinalidade | Validação | Não correspondências |
|---|---|---|---|---|
| livros ⟕ autores | `id_autor` | muitos para um | `validate="many_to_one"`, `indicator=True` | 0 |
| autores ⟕ auditoria por nome | `id_autor` | um para um | `validate="one_to_one"` | 0 |
| livros – livro_generos | `id_livro` | um para muitos (2 por livro) | não unida à base; teste de inclusão dos ids | 0 |

**Vínculo entre texto, metadado, indicador e fonte:** o título (texto curto),
os metadados e os indicadores de reconhecimento ficam na mesma linha do livro,
ligados por `id_livro` ao arquivo bruto e à página do Goodreads.

**Contagens antes/depois e riscos de multiplicação:** 10.454 livros antes e
depois da junção com autores. A junção ingênua com `livro_generos` produziria
20.908 linhas e dobraria a soma de avaliações; ela é bloqueada por
`validate="one_to_one"`.

As junções produzem a base integrada, mas o resultado ainda precisa demonstrar
como foi construído. Logs e testes ligam cada saída às entradas, parâmetros e
decisões que tornam o processo reproduzível.

## 6. Registro e testes

**Log de transformações:** tabela da U03-A03 (Notebook 02) e
`dados/derivados/log_execucao.json`, gravado a cada reconstrução com versões de
bibliotecas, hash da entrada, saídas e resultado dos testes.

**Testes de esquema, domínios, chaves e arquivos:** 12 testes automáticos em
`preparar_base.testar` (hash, colunas, unicidade, ausências, domínio binário,
notas entre 1 e 5, resenhas ≤ avaliações, atributos de autor constantes,
preservação de linhas na junção, correspondência total, unicidade de autores e
duas relações por livro). Resultado na célula seguinte.

**Como reconstruir a base desde os brutos:** executar
`python preparar_base.py` dentro de `projeto_goodreads/` ou a célula seguinte.

**Erros conhecidos e casos pendentes:** rótulo de gênero com erro sistemático
(1.274 livros duvidosos); um nome de autor com dois ids (William Gibson); 38
livros com contagem maior que a do autor; 29 páginas suspeitas; data de coleta
e critério de seleção desconhecidos.

In [1]:
# Localiza a pasta do projeto Goodreads (localmente ou no Colab, se a pasta for enviada para /content)
import sys
from pathlib import Path

CANDIDATOS = [Path("../projeto_goodreads"), Path("projeto_goodreads"), Path("/content/projeto_goodreads")]
PROJETO = next((c.resolve() for c in CANDIDATOS if (c / "preparar_base.py").exists()), None)
if PROJETO is None:
    raise FileNotFoundError(
        "Pasta projeto_goodreads não encontrada. Coloque-a ao lado das pastas unidade_0X "
        "ou, no Colab, envie-a para /content."
    )
if str(PROJETO) not in sys.path:
    sys.path.insert(0, str(PROJETO))
print("Projeto localizado:", (PROJETO / "preparar_base.py").exists())

from preparar_base import construir_base
import json
import pandas as pd

resultado = construir_base(gravar=True)
display(resultado["testes"])
print("Todos os testes passaram:", bool(resultado["testes"].all()))
print(json.dumps(json.loads((PROJETO / "dados" / "derivados" / "log_execucao.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=1)[:900])

Projeto localizado: True


hash_bruto_confere                   True
colunas_esperadas                    True
id_livro_unico                       True
sem_valores_ausentes_no_bruto        True
genero_autor_binario                 True
notas_entre_1_e_5                    True
resenhas_nao_excedem_avaliacoes      True
atributos_do_autor_constantes        True
juncao_preservou_livros              True
todos_os_livros_tem_autor            True
id_autor_unico_na_tabela_autores     True
relacoes_livro_genero_2_por_livro    True
Name: resultado, dtype: bool

Todos os testes passaram: True
{
 "entrada": "dados/brutos/livros_good_reads.csv",
 "sha256_entrada": "40effdd64def421fbd9b6665542ebfbf058ad3c6c902acfba96227763b9cfe64",
 "pandas": "2.3.3",
 "numpy": "2.5.3",
 "auditoria_genero_por_nome": "gender-guesser 0.4.0, dicionário de nomes de Jörg Michael (2007-2008); uso restrito à auditoria do rótulo",
 "saidas": [
  "auditoria_genero_nome.csv",
  "autores.csv",
  "base_analitica.csv",
  "livro_generos.csv",
  "livros.csv"
 ],
 "testes": {
  "hash_bruto_confere": true,
  "colunas_esperadas": true,
  "id_livro_unico": true,
  "sem_valores_ausentes_no_bruto": true,
  "genero_autor_binario": true,
  "notas_entre_1_e_5": true,
  "resenhas_nao_excedem_avaliacoes": true,
  "atributos_do_autor_constantes": true,
  "juncao_preservou_livros": true,
  "todos_os_livros_tem_autor": true,
  "id_autor_unico_na_tabela_autores": true,
  "relacoes_livro_genero_2_por_livro": true
 }
}


In [2]:
# Relatório de qualidade resumido da base derivada
base = resultado["base_analitica"]
pd.Series({
    "livros": len(base),
    "autores": base["id_autor"].nunique(),
    "livros_com_rotulo_duvidoso": int(base["rotulo_duvidoso"].sum()),
    "livros_E2_feminino": int(base["genero_E2_sem_duvidosos"].eq("feminino").sum()),
    "livros_E2_masculino": int(base["genero_E2_sem_duvidosos"].eq("masculino").sum()),
    "pais_padronizado_ausente": int(base["pais_padronizado"].isna().sum()),
    "paginas_suspeitas": int(base["paginas_suspeitas"].sum()),
    "contagem_incoerente": int(base["contagem_livro_maior_que_autor"].sum()),
})

livros                        10454
autores                        7780
livros_com_rotulo_duvidoso     1274
livros_E2_feminino             4638
livros_E2_masculino            4542
pais_padronizado_ausente         14
paginas_suspeitas                29
contagem_incoerente              38
dtype: int64

Com inventário, transformações, relações e testes documentados, já podemos
avaliar o pacote como um todo. A rubrica converte os requisitos metodológicos
em critérios comuns para autoavaliação.

## 7. Rubrica

| Critério | 0 | 1 | 2 | Minha nota/evidência |
|---|---|---|---|---|
| preservação e proveniência | ausente | parcial | auditável | 2 na preservação (bruto somente leitura, hash); proveniência externa incompleta, o que limita a nota geral a 1 |
| importação | implícita | parâmetros parciais | testada | 2: codificação, separador e colunas testados |
| limpeza | sobrescreve | regras incompletas | original + regra + teste | 2: nenhuma coluna original sobrescrita; regras e testes registrados |
| ausências/duplicatas | descartadas | diagnosticadas | justificadas | 2: ausências com razão; duplicatas revisadas e mantidas |
| junções | sem controle | chave declarada | cardinalidade auditada | 2: `validate` e `indicator` em todas as junções |
| integração | arquivos soltos | vínculos parciais | chaves verificadas | 2: cinco tabelas ligadas por ids testados |
| reprodutibilidade | manual | ordem parcial | reconstruível | 2: base reconstruída por um único comando, com log |

> **Nota sobre a revisão por pares.** Esta resolução foi produzida por uma única
> pessoa. Apresento uma **revisão crítica simulada**, guiada pelas perguntas do
> roteiro. Ela não substitui o parecer de um colega real.

## U03-A06 — 8. Entrega e revisão por pares

Entregue inventário, notebooks, base processável, tabelas relacionais, log,
relatório de qualidade e ficha de proveniência. O colega deve tentar localizar
uma regra não reversível, uma junção perigosa e um limite não documentado.

**Parecer recebido e mudanças:**
*Regra não reversível:* nenhuma coluna original foi sobrescrita, mas a
exclusão dos rótulos duvidosos em E2 é irreversível *dentro da análise*: quem
ler só o resultado de E2 não saberá que 1.274 livros ficaram de fora.
*Mudança:* toda tabela de resultado das Unidades 4 e 5 informará a
especificação e o número de livros usados.
*Junção perigosa:* juntar a tabela livro–gênero à base de livros duplicaria
avaliações. *Mudança:* a junção foi bloqueada com `validate` e demonstrada no
Notebook 03.
*Limite não documentado:* a auditoria por nome usa o dicionário de 2007–2008,
anterior a muitos autores do recorte. *Mudança:* acrescentei a data do
dicionário à coluna `ferramenta` e ao log.

**O que a Unidade 4 poderá explorar e o que ainda não deve concluir:**
Poderá explorar a distribuição das avaliações, das resenhas e da taxa de
resenhas por grupo, por período e por gênero literário, sempre com as três
especificações de gênero e com casos extremos inspecionados. Também poderá
explorar o vocabulário dos títulos. Não deve concluir que os grupos diferem na
população do Goodreads, que a diferença tem uma causa, que livros de um grupo
são melhores ou que o rótulo corrigido corresponde ao gênero real de cada
pessoa.